# DEMO 11: Star Schemas vs Pre-Joined Tables

In Power BI, a **star schema** with DAX measures "just works" — the relationship engine automatically traverses JOINs behind the scenes. In Databricks SQL, every JOIN must be **explicit**, which has performance and complexity implications for dashboards.

In this demo, you'll see:
- How a **star schema** query requires multiple JOINs (and what the execution plan looks like)
- How a **pre-joined Gold table** eliminates JOINs entirely (simpler + often faster)
- How a **view over the star schema** gives you the best of both worlds
- The difference between **BroadcastHashJoin** and **ShuffleExchange** in query plans

> **Note:** Run the setup cell below (Cell 2) to create the demo tables, then open the **SQL Editor** (click "SQL Editor" in the left sidebar) to run the remaining queries there.

---

In [0]:
%run ./demo_11_setup

## Part 1: The Star Schema Pattern

Our star schema has a central **fact table** surrounded by **dimension tables**:

```
                dim_customers (50 rows)
                     |
 dim_regions (8) -- fact_sales (3000 rows) -- dim_products (25)
```

In Power BI, you'd define relationships and DAX handles the JOINs implicitly. In Databricks SQL, you must write explicit JOINs every time.

**The question:** "Total revenue by region, product category, and customer segment for Q3 2024"

In Power BI, you'd drag fields from different tables onto a matrix visual and DAX would figure it out. In SQL, you need to JOIN all the tables yourself.

---

In [0]:
%sql
-- ============================================================
-- Star Schema: Query with multiple JOINs
-- ============================================================
-- This is what dashboard SQL looks like with a star schema.
-- Notice: 3 JOINs required just to get basic dimension attributes.

SELECT
  r.region_name,
  p.category AS product_category,
  c.segment AS customer_segment,
  COUNT(*) AS order_count,
  ROUND(SUM(f.quantity * f.unit_price * (1 - f.discount_pct)), 2) AS net_revenue
FROM fact_sales f
INNER JOIN dim_regions r ON f.region_id = r.region_id
INNER JOIN dim_products p ON f.product_id = p.product_id
INNER JOIN dim_customers c ON f.customer_id = c.customer_id
WHERE f.sale_date >= '2024-07-01'
  AND f.sale_date < '2024-10-01'
GROUP BY r.region_name, p.category, c.segment
ORDER BY net_revenue DESC;

## What Happens Under the Hood: Shuffles and Broadcasts

When Spark executes a JOIN, it needs data from both sides to be **co-located** (on the same worker). It has two strategies:

| Strategy | How It Works | Performance |
| --- | --- | --- |
| **BroadcastHashJoin** | Copy the small table to ALL workers | Fast — no shuffle of the large table |
| **SortMergeJoin** | Redistribute (shuffle) BOTH tables by the join key | Slow — massive data movement |

With our star schema, the dimension tables are small (8-50 rows), so Spark will likely **broadcast** them. But as dimensions grow or you add more JOINs, performance degrades.

Let's look at the query plan to see this in action:

---

In [0]:
%sql
-- ============================================================
-- EXPLAIN: See the JOINs and data movement
-- ============================================================
-- Look for:
-- - Multiple BroadcastHashJoin operators (one per dimension)
-- - BroadcastExchange operators (copying dim tables to workers)
-- - The scan on fact_sales with DataFilters for the date range

EXPLAIN
SELECT
  r.region_name,
  p.category AS product_category,
  c.segment AS customer_segment,
  COUNT(*) AS order_count,
  ROUND(SUM(f.quantity * f.unit_price * (1 - f.discount_pct)), 2) AS net_revenue
FROM fact_sales f
INNER JOIN dim_regions r ON f.region_id = r.region_id
INNER JOIN dim_products p ON f.product_id = p.product_id
INNER JOIN dim_customers c ON f.customer_id = c.customer_id
WHERE f.sale_date >= '2024-07-01'
  AND f.sale_date < '2024-10-01'
GROUP BY r.region_name, p.category, c.segment
ORDER BY net_revenue DESC;

## Part 2: The Pre-Joined Gold Table Pattern

The Databricks recommendation for dashboards: **pre-join the data upstream** into a wide, flat Gold table. The data engineering team maintains the star schema in Silver, and the Gold layer provides dashboard-ready datasets.

**Benefits:**
- Dashboard queries become simple `SELECT ... WHERE ... GROUP BY` (no JOINs)
- The query optimizer has less work to do at runtime
- Dashboard authors don't need to understand the relational model
- Performance is more predictable (no join strategy decisions at query time)

The **same business question** now becomes a single-table query:

---

In [0]:
%sql
-- ============================================================
-- Pre-Joined Gold Table: Same question, NO JOINs
-- ============================================================
-- Compare the simplicity! Same business answer, but:
-- - No JOINs
-- - No knowledge of the relational model needed
-- - Revenue is pre-calculated (net_revenue column)
-- - Date parts are pre-extracted (sale_quarter)

SELECT
  region_name,
  product_category,
  customer_segment,
  COUNT(*) AS order_count,
  ROUND(SUM(net_revenue), 2) AS total_revenue
FROM gold_sales_wide
WHERE sale_date >= '2024-07-01'
  AND sale_date < '2024-10-01'
GROUP BY region_name, product_category, customer_segment
ORDER BY total_revenue DESC;

In [0]:
%sql
-- ============================================================
-- EXPLAIN: See how simple the plan is without JOINs
-- ============================================================
-- Compare to the star schema EXPLAIN above:
-- - NO BroadcastHashJoin operators
-- - NO BroadcastExchange operators
-- - Just: Scan -> Filter -> Aggregate -> Sort
-- - This is a much simpler (and faster) execution plan

EXPLAIN
SELECT
  region_name,
  product_category,
  customer_segment,
  COUNT(*) AS order_count,
  ROUND(SUM(net_revenue), 2) AS total_revenue
FROM gold_sales_wide
WHERE sale_date >= '2024-07-01'
  AND sale_date < '2024-10-01'
GROUP BY region_name, product_category, customer_segment
ORDER BY total_revenue DESC;

## Part 3: Views — Best of Both Worlds

What if you want to keep the star schema for flexibility but still give dashboard authors a simple interface? **Create a view** over the star schema.

A view:
- Maintains the normalized star schema underneath (easy to maintain, no data duplication)
- Presents a flat, pre-joined interface to dashboard authors
- The JOIN logic is written once in the view definition, not repeated in every query

**Tradeoff:** The JOINs still execute every time the view is queried. For small-to-medium datasets, this is fine. For very large datasets with many concurrent dashboard users, a materialized Gold table may be better.

---

In [0]:
%sql
-- ============================================================
-- Create a VIEW that hides the star schema complexity
-- ============================================================
-- Dashboard authors query this view as if it were a flat table.
-- The JOIN logic lives here, not in every dashboard widget.

CREATE OR REPLACE VIEW v_sales_dashboard AS
SELECT
  f.sale_id,
  f.sale_date,
  YEAR(f.sale_date) AS sale_year,
  QUARTER(f.sale_date) AS sale_quarter,
  c.customer_name,
  c.segment AS customer_segment,
  c.loyalty_tier,
  p.product_name,
  p.category AS product_category,
  p.brand,
  r.region_name,
  r.country,
  f.quantity,
  f.unit_price,
  f.discount_pct,
  ROUND(f.quantity * f.unit_price * (1 - f.discount_pct), 2) AS net_revenue
FROM fact_sales f
INNER JOIN dim_customers c ON f.customer_id = c.customer_id
INNER JOIN dim_products p ON f.product_id = p.product_id
INNER JOIN dim_regions r ON f.region_id = r.region_id;

In [0]:
%sql
-- ============================================================
-- Query the view: dashboard authors see a flat table
-- ============================================================
-- Same business question again, but now the dashboard author
-- doesn't need to know about JOINs or the star schema.
-- The view handles all the complexity.

SELECT
  region_name,
  product_category,
  customer_segment,
  COUNT(*) AS order_count,
  ROUND(SUM(net_revenue), 2) AS total_revenue
FROM v_sales_dashboard
WHERE sale_date >= '2024-07-01'
  AND sale_date < '2024-10-01'
GROUP BY region_name, product_category, customer_segment
ORDER BY total_revenue DESC;

## Part 4: Side-by-Side — Query Complexity Comparison

Look at the three approaches to the same business question:

| Approach | Lines of SQL | JOINs Required | Performance |
| --- | --- | --- | --- |
| **Star Schema** | 15+ | 3 explicit JOINs | Good (broadcast joins on small dims) |
| **Gold Table** | 8 | 0 | Best (simple scan + aggregate) |
| **View** | 8 (view hides 15+) | 0 visible (3 hidden) | Same as star schema underneath |

**The Databricks recommendation:**
- **Silver layer:** Maintain the star schema (normalized, flexible, DRY)
- **Gold layer:** Create pre-joined tables or views for specific dashboard use cases
- **Dashboard:** Point at Gold — keep queries as simple as possible

---

In [0]:
%sql
-- ============================================================
-- Dashboard-Ready Queries: Simple and Fast
-- ============================================================
-- These are the kind of queries a dashboard widget should use.
-- No JOINs, clear column names, appropriate filters.

-- Widget 1: Revenue by region (current year)
SELECT region_name, ROUND(SUM(net_revenue), 2) AS total_revenue
FROM gold_sales_wide
WHERE sale_year = 2024
GROUP BY region_name
ORDER BY total_revenue DESC;

-- Widget 2: Top 5 brands by order count
SELECT brand, COUNT(*) AS order_count
FROM gold_sales_wide
WHERE sale_year = 2024
GROUP BY brand
ORDER BY order_count DESC
LIMIT 5;

-- Widget 3: Monthly revenue trend
SELECT sale_month, ROUND(SUM(net_revenue), 2) AS monthly_revenue
FROM gold_sales_wide
WHERE sale_year = 2024
GROUP BY sale_month
ORDER BY sale_month;

---

## Summary

| Concept | Key Takeaway |
| --- | --- |
| **Star Schema** | Works in Databricks but requires explicit JOINs in every query |
| **Pre-Joined Gold Table** | Eliminates JOINs — simpler queries, more predictable performance |
| **View over Star Schema** | Hides complexity while maintaining normalized source |
| **BroadcastHashJoin** | Small dims broadcast to all workers (fast, but still adds plan complexity) |
| **ShuffleExchange** | Data redistributed across workers (expensive — avoid if possible) |

**Power BI vs Databricks:**
- In Power BI, the relationship engine handles JOINs implicitly — you never see them
- In Databricks, you either write JOINs yourself or pre-join the data upstream
- The Gold layer pattern is how Databricks achieves the same "drag and drop" simplicity

**Practical advice for dashboard design:**
1. Ask your data engineering team for a **Gold table or view** for your dashboard
2. Keep dashboard SQL to **3–5 lines** — SELECT, FROM, WHERE, GROUP BY
3. If your query has JOINs, consider whether a view should handle them instead

---

**Next:** We'll look at how **Metric Views** can provide reusable business metrics — the Databricks equivalent of DAX measures.

---

## Summary

| Concept | Key Takeaway |
| --- | --- |
| **Star Schema** | Works in Databricks but requires explicit JOINs in every query |
| **Pre-Joined Gold Table** | Eliminates JOINs — simpler queries, more predictable performance |
| **View over Star Schema** | Hides complexity while maintaining normalized source |
| **BroadcastHashJoin** | Small dims broadcast to all workers (fast, but still adds plan complexity) |
| **ShuffleExchange** | Data redistributed across workers (expensive — avoid if possible) |

**Power BI vs Databricks:**
- In Power BI, the relationship engine handles JOINs implicitly — you never see them
- In Databricks, you either write JOINs yourself or pre-join the data upstream
- The Gold layer pattern is how Databricks achieves the same "drag and drop" simplicity

**Practical advice for dashboard design:**
1. Ask your data engineering team for a **Gold table or view** for your dashboard
2. Keep dashboard SQL to **3–5 lines** — SELECT, FROM, WHERE, GROUP BY
3. If your query has JOINs, consider whether a view should handle them instead

---

**Next:** We'll look at how **Metric Views** can provide reusable business metrics — the Databricks equivalent of DAX measures.